In [0]:
from pyspark.sql import Row
import json
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import StringType, MapType

# read the bronze layer
bronze_df = spark.read.format("delta").load(
    "abfss://lyon-data@stlyonloyersdev.dfs.core.windows.net/bronze/delta/loyer"
)

In [0]:
# ouvert le structure json

import json
import pandas as pd
from pyspark.sql import functions as F


pdf_bronze = bronze_df.toPandas()

records = []
for idx, row in pdf_bronze.iterrows():
    codeiris = str(row.get("codeiris"))
    zonage = str(row.get("zonage"))
    commune = str(row.get("commune"))
    valeurs_str = row.get("valeurs")
    
    if pd.notna(valeurs_str):
        try:
            valeurs_dict = json.loads(valeurs_str)
            for nb_pieces, years_dict in valeurs_dict.items():
                for period, furniture_dict in years_dict.items():
                    for is_furnished, metrics in furniture_dict.items():
                        records.append({
                            "codeiris": codeiris,
                            "zonage": zonage,
                            "commune": commune,
                            "nb_pieces": int(nb_pieces),
                            "construction_period": period,
                            "is_furnished": is_furnished,
                            "loyer_reference": float(metrics.get("loyer_reference", 0.0)),
                            "loyer_ref_majore": float(metrics.get("loyer_reference_majore", 0.0)),
                            "loyer_ref_minore": float(metrics.get("loyer_reference_minore", 0.0)),
                            "majoration_unitaire": float(metrics.get("majoration_unitaire", 0.0))
                        })
        except Exception as e:
            continue

# to Spark DataFrame
silver_df = spark.createDataFrame(pd.DataFrame(records))

In [0]:
silver_df.show(10, truncate=False)

In [0]:
# vérifier les valeur null

from pyspark.sql import functions as F

silver_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
]).show()

In [0]:
# Vérifier les doublon
silver_df.groupBy(
    "codeiris",
    "nb_pieces",
    "construction_period",
    "is_furnished"
).count().filter(
    F.col("count") > 1
).show()

In [0]:
SILVER_PATH = "abfss://lyon-data@stlyonloyersdev.dfs.core.windows.net/silver/delta/loyers"

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(SILVER_PATH)
)

save to catalog

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbx_lyonloyer_dev.silver.loyers")

In [0]:
%sql

SELECT *
FROM dbx_lyonloyer_dev.silver.loyers;